In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
%pwd

'/projectnb/batmanlab/mragoza/lung-project/notebooks/copdgene'

In [3]:
import sys, os
import pandas as pd

def add_path(p: str):
    if p not in sys.path:
        sys.path.append(p)

add_path(os.environ['LP_ROOT'])
import project

add_path(os.environ['PROJECT'] + '/' + 'param_search')
import param_search as ps

ps.set_backend('sge')
ps.set_verbose(False)

# Gather examples

In [4]:
from pathlib import Path
#data_root = Path(os.environ['LP_ROOT'] / 'data' / 'COPDGene'
data_root = Path(os.environ['PRIVATE']) / 'data' / 'COPDGene'
for p in data_root.iterdir():
    print(p)

/restricted/projectnb/batmanlab/mragoza/data/COPDGene/Processed
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/sample1000_2025-07-22.txt
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/sample1000_2025-07-22.csv
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/subject_files.txt
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/Images
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/sample1000_2025-07-17.csv
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/DISK_USAGE
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/ClinicalData


In [5]:
subject_file = data_root / 'sample1000_2025-07-22.csv'
subject_list = list(pd.read_csv(subject_file, sep='\t').sid)
subject_list

['16514P',
 '20748Q',
 '11007Z',
 '14771Z',
 '13651K',
 '15900P',
 '15312Y',
 '21042H',
 '21877G',
 '10127E',
 '23577E',
 '13816Q',
 '21559S',
 '25335Q',
 '23023N',
 '10572Z',
 '10887Y',
 '21410K',
 '17920F',
 '14684E',
 '16132B',
 '17862R',
 '11746L',
 '25728J',
 '20609C',
 '16637F',
 '13297S',
 '25767T',
 '25695U',
 '21572K',
 '15078Q',
 '15297C',
 '14857J',
 '13460D',
 '18840M',
 '12506W',
 '11498S',
 '21611U',
 '15088T',
 '12831H',
 '19612E',
 '19784H',
 '10217F',
 '23037Y',
 '25130Y',
 '15623P',
 '19027T',
 '14380K',
 '10212V',
 '16060C',
 '19356M',
 '22357L',
 '25923H',
 '21189L',
 '24608U',
 '15204V',
 '11743F',
 '11879E',
 '12477P',
 '10815Z',
 '17790S',
 '15489L',
 '11850G',
 '13034M',
 '17255W',
 '14632L',
 '26066U',
 '18184E',
 '15532M',
 '19558Y',
 '20640W',
 '21741H',
 '18015H',
 '17137Q',
 '12422Q',
 '15699W',
 '21933Q',
 '18397V',
 '14550J',
 '20703U',
 '21004Z',
 '24581A',
 '24331D',
 '17505T',
 '15894U',
 '16977D',
 '18935X',
 '19410S',
 '18515B',
 '21399W',
 '21899Q',

In [17]:
base_dir = '2026-08-08_preprocess'

template = '''\
#!/bin/bash -l
#$ -N {job_name}
#$ -P batmanlab
#$ -pe omp 4
#$ -l gpus=1
#$ -l gpu_memory=16G
#$ -l h_rt=12:00:00
set -eo pipefail

mamba activate $PROJECT/mambaforge/envs/warp

export PYTHONPATH=$LP_ROOT:$PROJECT:$PYTHOPATH

python $LP_ROOT/scripts/preprocess.py {config} \\
    --set dataset.name={data_name} \\
    --set dataset.root={data_root} \\
    --set dataset.examples.subjects=[{subject}] \\
    --set dataset.examples.variant={variant} \\
    --set dataset.examples.pipeline_tags.material_properties={mat_tag} \\
    --set dataset.examples.pipeline_tags.forward_simulation={fwd_tag} \\
    --set preprocessing.forward_simulation.pde_solver.material_type={mat_type}

'''
name_format = 'job_{params_hash}'

grid = ps.param_grid(
    config='2026-08-08_config.yaml',
    data_name='COPDGene',
    data_root=str(data_root),
    subject=subject_list,
    mat_tag='bela',
    fwd_tag='linear',
    mat_type='linear',
    variant='2026-08-08'
)

ps.param_grid(  # TODO
    config='2026-08-08_config.yaml',
    data_name='COPDGene',
    data_root=str(data_root),
    subject=subject_list,
    mat_tag='bela',
    fwd_tag='stvk',
    mat_type='stvk',
    variant='2026-08-08'
)
len(grid)

1000

In [18]:
%autoreload
try:
    jobs = ps.setup(base_dir, template, name_format, grid, overwrite=True)
except OSError:
    jobs = ps.load(base_dir)

jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params_json,params_hash,params.config,params.data_name,params.data_root,params.subject,params.mat_tag,params.fwd_tag,params.mat_type,params.variant
0,job_ead9ab1e5bc03c66,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",ead9ab1e5bc03c66,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,16514P,bela,linear,linear,2026-08-08
1,job_96ef0272ef81ee7e,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",96ef0272ef81ee7e,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,20748Q,bela,linear,linear,2026-08-08
2,job_4d5d759871906077,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",4d5d759871906077,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,11007Z,bela,linear,linear,2026-08-08
3,job_62c1bdeb21500ce0,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",62c1bdeb21500ce0,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,14771Z,bela,linear,linear,2026-08-08
4,job_1a856f1cecae8f51,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",1a856f1cecae8f51,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,13651K,bela,linear,linear,2026-08-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,job_22f2adacb02a421c,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",22f2adacb02a421c,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,20519B,bela,linear,linear,2026-08-08
996,job_b1e67f498dc392b6,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",b1e67f498dc392b6,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,12294H,bela,linear,linear,2026-08-08
997,job_2f617f39d23060c8,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",2f617f39d23060c8,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,23123R,bela,linear,linear,2026-08-08
998,job_bef061b64dfcfc4e,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",bef061b64dfcfc4e,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,16546C,bela,linear,linear,2026-08-08


In [19]:
jobs = ps.submit(jobs)

In [8]:
%autoreload
jobs = ps.recover(jobs)
jobs = ps.status(jobs)
jobs = ps.history(jobs)
jobs = ps.collect(jobs)
jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.mat_type,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
0,jobead9ab1e5bc03c66,SUBMITTED,1,7159234,None,None,,usage: preprocess.py [-h] [--set KEY=VAL] conf...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
1,job96ef0272ef81ee7e,SUBMITTED,1,7159235,None,None,,usage: preprocess.py [-h] [--set KEY=VAL] conf...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
2,job4d5d759871906077,SUBMITTED,1,7159236,None,None,,usage: preprocess.py [-h] [--set KEY=VAL] conf...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
3,job62c1bdeb21500ce0,SUBMITTED,1,7159237,None,None,,usage: preprocess.py [-h] [--set KEY=VAL] conf...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
4,job1a856f1cecae8f51,SUBMITTED,1,7159238,None,None,,usage: preprocess.py [-h] [--set KEY=VAL] conf...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,job22f2adacb02a421c,SUBMITTED,1,7160229,None,None,,usage: preprocess.py [-h] [--set KEY=VAL] conf...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
996,jobb1e67f498dc392b6,SUBMITTED,1,7160230,None,None,,usage: preprocess.py [-h] [--set KEY=VAL] conf...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
997,job2f617f39d23060c8,SUBMITTED,1,7160231,None,None,,usage: preprocess.py [-h] [--set KEY=VAL] conf...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>
998,jobbef061b64dfcfc4e,SUBMITTED,1,7160232,None,None,,usage: preprocess.py [-h] [--set KEY=VAL] conf...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,linear,2026-08-08,NaN,NaN,NaN,NaN,NaN,False,<NA>,<NA>


In [9]:
jobs.groupby('job_state').count()

,job_name,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,script_path,...,params.mat_type,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
job_state,,,,,,,,,,,,,,,,,,,,,
SUBMITTED,1000,1000,1000,0,0,1000,1000,1000,1000,1000,...,1000,1000,0,0,0,0,0,1000,0,0


In [10]:
# classify error messages
for j, row in jobs.iterrows():
    if pd.isnull(row.stderr):
        print(f'{j}: No stderr file')
    elif not str(row.stderr).strip():
        print(f'{j}: No error message')
    elif 'Non-positive density' in row.stderr:
        print(f'{j}: Non-positive density')
    elif 'CUDA out of memory' in row.stderr or ('Failed to allocate' in row.stderr and 'bytes on device \'cuda' in row.stderr):
        print(f'{j}: GPU out of memory')
    elif 'Exuding' in row.stderr or ('Moving #' in row.stderr and 'addr:' in row.stderr):
        print(f'{j}: pygalmesh output')
    elif 'construct initial points' in row.stderr:
        print(f'{j}: pygalmesh error?')
    else:
        print(f'{j}: Uncategorized')
        raise RuntimeError(f'Uncategorized stderr for job index {j}:\n{row.stderr}')

0: Uncategorized


RuntimeError: Uncategorized stderr for job index 0:
usage: preprocess.py [-h] [--set KEY=VAL] config
preprocess.py: error: unrecognized arguments:  


In [18]:
print(jobs.iloc[261].stdout)

Loading /restricted/projectnb/batmanlab/mragoza/data/COPDGene/Processed/2026-08-08/26071N/masks/26071N_INSP_iso_tsvf_domain.nii.gz
Loading /restricted/projectnb/batmanlab/mragoza/data/COPDGene/Processed/2026-08-08/26071N/masks/26071N_INSP_iso_tsvf/lung_airways.nii.gz
Loading /restricted/projectnb/batmanlab/mragoza/data/COPDGene/Processed/2026-08-08/26071N/masks/26071N_INSP_iso_tsvf/consolidation.nii.gz
Loading /restricted/projectnb/batmanlab/mragoza/data/COPDGene/Processed/2026-08-08/26071N/masks/26071N_INSP_iso_tsvf/lung_arteries.nii.gz
Loading /restricted/projectnb/batmanlab/mragoza/data/COPDGene/Processed/2026-08-08/26071N/masks/26071N_INSP_iso_tsvf/lt-950.nii.gz
Loading /restricted/projectnb/batmanlab/mragoza/data/COPDGene/Processed/2026-08-08/26071N/masks/26071N_INSP_iso_tsvf/lung_veins.nii.gz
Loading /restricted/projectnb/batmanlab/mragoza/data/COPDGene/Processed/2026-08-08/26071N/masks/26071N_INSP_iso_tsvf/lt-850.nii.gz
Loading /restricted/projectnb/batmanlab/mragoza/data/COPDGe

In [25]:
jobs.loc[:, 'job_id'] = pd.NA

In [28]:
%autoreload
jobs = ps.submit(jobs)
jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params_json,params_hash,params.config,params.data_name,params.data_root,params.subject,params.mat_tag,params.fwd_tag,params.mat_type,params.variant
0,jobead9ab1e5bc03c66,SUBMITTED,1,7159234,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",ead9ab1e5bc03c66,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,16514P,bela,linear,linear,2026-08-08
1,job96ef0272ef81ee7e,SUBMITTED,1,7159235,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",96ef0272ef81ee7e,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,20748Q,bela,linear,linear,2026-08-08
2,job4d5d759871906077,SUBMITTED,1,7159236,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",4d5d759871906077,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,11007Z,bela,linear,linear,2026-08-08
3,job62c1bdeb21500ce0,SUBMITTED,1,7159237,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",62c1bdeb21500ce0,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,14771Z,bela,linear,linear,2026-08-08
4,job1a856f1cecae8f51,SUBMITTED,1,7159238,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",1a856f1cecae8f51,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,13651K,bela,linear,linear,2026-08-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,job22f2adacb02a421c,SUBMITTED,1,7160229,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",22f2adacb02a421c,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,20519B,bela,linear,linear,2026-08-08
996,jobb1e67f498dc392b6,SUBMITTED,1,7160230,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",b1e67f498dc392b6,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,12294H,bela,linear,linear,2026-08-08
997,job2f617f39d23060c8,SUBMITTED,1,7160231,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",2f617f39d23060c8,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,23123R,bela,linear,linear,2026-08-08
998,jobbef061b64dfcfc4e,SUBMITTED,1,7160232,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,"{""config"": ""2026-08-08_config.yaml"", ""data_nam...",bef061b64dfcfc4e,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,16546C,bela,linear,linear,2026-08-08
